## 11.01节练习参考答案

### 环境准备

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F

from src.pto_layers import PyPTOLinear, PyPTOReLU

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch11/ch11)，并在此基础上补充了 PyPTO 的实现。  

### 练习11.1.1
考虑一个简单的MLP，它有一个隐藏层，比如，隐藏层中维度为d 和一个输出。证明对于任何局部最小值，至少有d！ 个等效方案。

**解答：**

In [3]:
class Net(nn.Module):
  def __init__(self, dim, out):
    super(Net, self).__init__()
    self.hidden = nn.Linear(dim, dim)
    self.out_layer = nn.Linear(dim, out)

  def forward(self, x):
    x = self.hidden(x)
    x = self.out_layer(x)
    return x

d = 10
out = 1
net = Net(d, out)
input = torch.randn(d)

output = net(input)

output

tensor([-0.4132], grad_fn=<ViewBackward0>)

本题为隐藏层神经元的排列对称性问题。

设隐藏层输出为 $\boldsymbol{h}=\sigma(W\boldsymbol{x}+\boldsymbol{b})$，输出为 $y=\boldsymbol{w}^\top\boldsymbol{h}+b'$。对 $\{1,\dots,d\}$ 的任意置换 $\pi$，记 $P_\pi$ 为对应置换矩阵（$P_\pi^\top P_\pi=I$），令

$$W'=P_\pi W,\qquad \boldsymbol{b}'=P_\pi\boldsymbol{b},\qquad \boldsymbol{w}'=P_\pi\boldsymbol{w}.$$

则

$$(\boldsymbol{w}')^\top\sigma(W'\boldsymbol{x}+\boldsymbol{b}')=(P_\pi\boldsymbol{w})^\top\sigma(P_\pi W\boldsymbol{x}+P_\pi\boldsymbol{b})=\boldsymbol{w}^\top P_\pi^\top P_\pi\,\sigma(W\boldsymbol{x}+\boldsymbol{b})=\boldsymbol{w}^\top\boldsymbol{h}=y.$$

故 $(W',\boldsymbol{b}',\boldsymbol{w}',b')$ 与原参数计算同一函数，损失相同，为等价的局部最小值。$d$ 个神经元共有 $d!$ 种置换，因此任一局部最小值至少对应 $d!$ 个等价解。

**PyPTO 版**

In [6]:
class Net(nn.Module):
  def __init__(self, dim, out):
    super(Net, self).__init__()
    self.hidden = PyPTOLinear(dim, dim)
    self.out_layer = PyPTOLinear(dim, out)

  def forward(self, x):
    x = self.hidden(x)
    x = self.out_layer(x)
    return x

d = 10
out = 1
net = Net(d, out)
input = torch.randn(d).npu()

output = net(input)

output

tensor([0.9036], device='npu:0', grad_fn=<ViewBackward0>)

### 练习11.1.2
假设我们有一个对称随机矩阵$M$，其中条目$M_{ij}=M_{ji}$各自从某种概率分布$P_{ij}$中抽取。此外，假设$p_{ij}=p_{ji}$，即分布是对称的（详情请参见(Wigner, 1958)）。

1. 证明特征值的分布也是对称的。也就是说，对于任何特征向量$\lambda$，关联的特征值满足$P(\lambda>0)=P(\lambda<0)$。

2. 为什么以上没有暗示$P(\lambda>0)=0.5$

**解答：**

1. 由每个 $M_{ij}$ 的分布关于 $0$ 对称，知 $-M$ 与 $M$ 同分布（$M\stackrel{d}{=}-M$）。$-M$ 的特征值为 $\{-\lambda_i\}$（$M$ 特征值取反），故特征值分布关于原点对称：

$$P(\lambda>0)=P(\lambda<0).$$

2. 特征值可等于 $0$。由 $P(\lambda>0)+P(\lambda<0)+P(\lambda=0)=1$ 及对称性 $P(\lambda>0)=P(\lambda<0)$，

$$P(\lambda>0)=\frac{1-P(\lambda=0)}{2}\le 0.5,$$

当矩阵奇异（存在零特征值）时取严格不等号。

### 练习11.1.3
你能想到深度学习优化还涉及哪些其他挑战？

**解答：**

梯度消失，梯度爆炸，局部最优解，鞍点，loss不收敛，梯度悬崖...

### 练习11.1.4
假设你想在（真实的）鞍上平衡一个（真实的）球。

1. 为什么这很难？

2. 能利用这种效应来优化算法吗？

**解答：**

(1)为什么这么难？

1. 考虑球和鞍的几何形状和性质。鞍是一个具有曲率和变化的表面，而球是一个固体物体。

2. 球在鞍上的平衡涉及到重力、重心位置以及鞍的形状对球的支撑和稳定性的影响。 由于鞍的形状，球在鞍上的平衡点通常是一个不稳定的平衡点，稍微有一点扰动就可能使球失去平衡。 球的重心位置很容易偏离鞍的平衡点，这会导致球在鞍上的平衡非常困难。

* 将一个球平衡在一个鞍上的困难可以归因于一个物理效应，即不稳定性效应。
不稳定性效应是指系统在某个平衡点附近的微小扰动会引起系统远离平衡点的现象。在这个问题中，鞍是一个不稳定的平衡点，即球在鞍的顶点上的平衡点是不稳定的。这意味着，即使球在鞍的顶点上保持静止，微小的扰动或者偏移都会导致球失去平衡，滚落到鞍的一侧。

这种不稳定性效应是由鞍的形状和球的重心位置的限制所导致的。由于鞍的形状，球在鞍上的平衡点非常狭窄，稍微有一点扰动就可能使球失去平衡。同时，球的重心位置很容易偏离鞍的平衡点，由于重力的作用，球的重心位置通常会偏向鞍的一侧，而不是位于鞍的顶点上。这使得球在鞍上的平衡非常困难。

因此，将一个球平衡在一个鞍上的困难可以归因于不稳定性效应，即微小的扰动会导致球失去平衡，滚落到鞍的一侧。

(2)能利用这种效应来优化算法吗？

不稳定性在深度学习的优化种，即随机性和扰动性。

* 利用随机型：网络层的Dropout随机丢弃神经元；神经元权重随机初始化；
* 利用扰动性：网络加入噪声；mask机制；VAE中的denoise；diffuison
* 模拟退火，遗传算法